In [24]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.chat_models.tongyi import ChatTongyi

@tool(description="获取体重，返回整数，单位是 kg")
def get_weight() -> int:
    return 90

@tool(description="获取身高，返回整数，单位是 cm")
def get_height() -> int:
    return 172

agent = create_agent(
    model=ChatTongyi(model="qwen3-max"),
    tools=[get_weight, get_height],
    system_prompt="你是一个严格准许 ReAct 框架的智能体，必须按照：思考->行动->观察->再思考，这个流程来解决问题" \
    "并且每次智能思考并调用一个工具，不可以单词调用多个工具，" \
    "告诉我你的思考过程，工具的调用原因"
)

for chunk in agent.stream(
    {"messages": [{"role":"user", "content":"计算我的 BMI"}]},
    stream_mode="values"
):
    latest_message = chunk['messages'][-1]
    if latest_message.content:
        print(type(latest_message).__name__, latest_message.content)

    try:
        if latest_message.tool_calls:
            print(f"工具调用：{[tc['name'] for tc in latest_message.tool_calls] }")
    except AttributeError as e:
        pass




HumanMessage 计算我的 BMI
AIMessage 要计算 BMI（身体质量指数），我需要知道你的体重（kg）和身高（cm）。BMI 的计算公式是：

$$
\text{BMI} = \frac{\text{体重 (kg)}}{(\text{身高 (m)})^2}
$$

因此，我首先需要获取你的体重。


工具调用：['get_weight']
ToolMessage 90
AIMessage 我已经获取到你的体重是 90 kg。接下来，我需要获取你的身高（cm），以便计算 BMI。


工具调用：['get_height']
ToolMessage 172
AIMessage 我已经获取到你的身高是 172 cm。现在可以计算 BMI 了。

将身高从厘米转换为米：  
$$
172 \, \text{cm} = 1.72 \, \text{m}
$$

代入 BMI 公式：  
$$
\text{BMI} = \frac{90}{(1.72)^2} = \frac{90}{2.9584} \approx 30.42
$$

你的 BMI 约为 **30.42**，属于 **肥胖** 范围（根据世界卫生组织标准：BMI ≥ 30 为肥胖）。建议关注健康饮食和适当运动！


In [ ]:
from langchain_core.tools import tool
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import after_agent, after_model, before_agent, before_model, wrap_model_call
from langchain_community.chat_models.tongyi import ChatTongyi
from langgraph.runtime import Runtime

@tool(description="获取天气，传入城市名称，返回字符串")
def get_weather(city) -> str:
    return f"{city}天气：晴天"


@before_agent
def log_before_agent(state: AgentState, runtime:Runtime) -> None:

    print(f"[before agent] agent 启动，并附带{len(state['messages'])} 条消息")


@after_agent
def log_after_agent(state: AgentState, runtime:Runtime) -> None:
    print(f"[after agent] agent 启动，并附带{len(state['messages'])} 条消息")

@before_model
def log_before_model(state: AgentState, runtime:Runtime) -> None:
    print(f"[before model] model 启动，并附带{len(state['messages'])} 条消息")

@after_model
def log_after_model(state: AgentState, runtime:Runtime) -> None:
    print(f"[after model] model 启动，并附带{len(state['messages'])} 条消息")


@wrap_model_call
def model_call_hook(request, handler):
    print("model is running")
    return handler(request)

@rap_tool_call
def monitor_tool(request, handler):
    print(f"tool running: {request.tool_call["name"]}")
    print(f"parameter of tool: {request.tool_call["args"]}")
    return handler(request)


